In [1]:
!pip install --trusted-host pypi.org --trusted-host files.pythonhosted.org scikit-posthocs


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.stats.multitest as multi

# ==========================================
# 1. Configuration and Paths
# ==========================================

# Base project directory
BASE_PATH = 'E:\\Projetos\\ABM-WP' 

# Input Files
SCENARIO_FILE = 'consumo_previsto_todos_cenarios.csv'
LSTM_FILE = 'previsoes_futuras_2025_2035.csv'

# Directory Setup
INPUT_DIR_RES = os.path.join(BASE_PATH, 'resultados')
INPUT_DIR_MODELS = os.path.join(BASE_PATH, 'modelos IA')
INPUT_DIR_INC = os.path.join(BASE_PATH, 'includes')

# Ensure directories exist
for folder in [INPUT_DIR_RES]:
    if not os.path.exists(folder):
        os.makedirs(folder)

# Style configuration
plt.style.use('seaborn-v0_8-white')
sns.set_palette("husl")

def sum_yearly(df, date_col, value_cols):
    """ Helper function to aggregate monthly data into annual sums """
    df_year = df.copy()
    df_year['Year'] = df_year[date_col].dt.year
    return df_year.groupby('Year')[value_cols].sum().reset_index()


def main():
    print("--- Starting Parametric Analysis: Scenarios vs LSTM (Tukey HSD) ---")

    # ==========================================
    # 2. Data Loading and Preparation
    # ==========================================
    path_scenarios = os.path.join(INPUT_DIR_RES, SCENARIO_FILE)
    if not os.path.exists(path_scenarios):
        path_scenarios = os.path.join(INPUT_DIR_INC, SCENARIO_FILE)
    
    df_scenarios = pd.read_csv(path_scenarios, sep=';', decimal=',')
    
    # Handle dates
    if 'Mes' in df_scenarios.columns and 'Ano' in df_scenarios.columns:
        df_scenarios['Date'] = pd.to_datetime(df_scenarios['Ano'].astype(str) + '-' + 
                                            df_scenarios['Mes'].astype(str) + '-01')
    elif 'Mes_Ano' in df_scenarios.columns:
        df_scenarios['Date'] = pd.to_datetime(df_scenarios['Mes_Ano'], format='%m/%Y')
    
    scenario_cols = [c for c in df_scenarios.columns if c not in ['Mes', 'Ano', 'Mes_Ano', 'Date']]
    df_scenarios = df_scenarios[df_scenarios['Date'] <= pd.to_datetime('2034-12-31')]

    # Load LSTM
    path_lstm = os.path.join(INPUT_DIR_MODELS, LSTM_FILE)
    df_lstm = pd.read_csv(path_lstm, parse_dates=['Data'])
    df_lstm = df_lstm.rename(columns={'Data': 'Date', 'Valor Previsto': 'LSTM'})
    df_lstm = df_lstm[df_lstm['Date'] <= pd.to_datetime('2034-12-31')]

    # Combine and aggregate
    df_combined = pd.merge(df_scenarios, df_lstm[['Date', 'LSTM']], on='Date', how='left')
    all_series = scenario_cols + ['LSTM']
    df_annual = sum_yearly(df_combined, 'Date', all_series)

    # ==========================================
    # 3. Statistical Testing (ANOVA & Tukey)
    # ==========================================
    print("\n[1] PARAMETRIC ANALYSIS (ANOVA)")
    full_data_list = [df_annual[col] for col in all_series]
    f_val, p_anova = stats.f_oneway(*full_data_list)
    
    print(f"ANOVA Results: F = {f_val:.4f}, p = {p_anova:.4f}")

    if p_anova < 0.05:
        # Prepare data for Tukey Post-Hoc
        values, groups = [], []
        for col in all_series:
            values.extend(df_annual[col].tolist())
            group_label = 'LSTM' if col == 'LSTM' else col.split('_')[0]
            groups.extend([group_label] * len(df_annual))
            
        tukey = pairwise_tukeyhsd(endog=values, groups=groups, alpha=0.05)
        tukey_df = pd.DataFrame(data=tukey.summary().data[1:], columns=tukey.summary().data[0])
        
        # Filter: Comparisons involving LSTM
        tukey_lstm = tukey_df[(tukey_df['group1'] == 'LSTM') | (tukey_df['group2'] == 'LSTM')].copy()
        
        # Standardize view: Group A = LSTM, Group B = Scenario
        def standardize(row):
            if row['group2'] == 'LSTM':
                return pd.Series([row['group2'], row['group1'], -row['meandiff'], row['p-adj'], row['reject']])
            return pd.Series([row['group1'], row['group2'], row['meandiff'], row['p-adj'], row['reject']])

        tukey_lstm[['g1', 'g2', 'diff', 'padj', 'rej']] = tukey_lstm.apply(standardize, axis=1)
        tukey_final = tukey_lstm[['g2', 'diff', 'padj', 'rej']].rename(
            columns={'g2': 'Scenario', 'diff': 'Mean_Diff', 'padj': 'p_adj', 'rej': 'Significant'}
        ).sort_values(by='p_adj')
        
        # ==========================================
        # 4. Printing and Exporting Results
        # ==========================================
        
        # PRINT TO SCREEN
        print("\n" + "="*70)
        print("TUKEY HSD POST-HOC RESULTS (Baseline: LSTM)")
        print("="*70)
        print(tukey_final.to_string(index=False, formatters={
            'Mean_Diff': '{:,.2f}'.format,
            'p_adj': '{:.4f}'.format
        }))
        print("="*70)

        # Save TXT (Results folder)
        txt_path = os.path.join(INPUT_DIR_RES, 'tukey_lstm_comparison_summary.txt')
        tukey_final.to_csv(txt_path, sep='\t', index=False)
        
    else:
        print("ANOVA not significant. Skipping Tukey HSD.")

if __name__ == "__main__":
    main()

--- Starting Parametric Analysis: Scenarios vs LSTM (Tukey HSD) ---

[1] PARAMETRIC ANALYSIS (ANOVA)
ANOVA Results: F = 11.7585, p = 0.0000

TUKEY HSD POST-HOC RESULTS (Baseline: LSTM)
Scenario  Mean_Diff  p_adj  Significant
   CVIII 202,503.46 0.0000         True
     CXV 164,156.28 0.0000         True
    CXIV 163,128.50 0.0000         True
     CIV 198,073.18 0.0000         True
     CIX 203,317.43 0.0000         True
      CV 202,603.90 0.0000         True
     CVI 203,417.88 0.0000         True
    CVII 197,972.87 0.0000         True
   CXIII 159,759.51 0.0000         True
     CXI  72,908.81 0.5058        False
    CXII  72,730.87 0.5105        False
      CX  72,055.14 0.5285        False
  CXVIII  69,501.67 0.5966        False
   CXVII  69,186.22 0.6050        False
      CI  68,806.86 0.6150        False
    CXVI  68,037.49 0.6353        False
    CIII  62,191.78 0.7784        False
     CII  49,268.40 0.9645        False
